# Lesson 15 · Gaussian elimination — working backward from effect to cause, and the three fates of Ax = b

**Learning goals**
1. Write a system of linear equations as an augmented matrix `[A|b]`, and get comfortable with the matrix form of the equation `Ax = b`
2. Master the three elementary row operations (swap ER1, scale ER2, eliminate ER3) and carry them out by hand on concrete examples
3. Use `rank(A)` and `rank([A|b])` to classify a system's solutions: a unique solution, infinitely many, or none
4. Implement a `classify_system(A, b)` function that returns `'unique'`/`'many'`/`'none'`, and check it on three extra worked examples
5. (Picked up again in the wrap-up) Know that an overdetermined system can be handled with least squares, `lstsq` — Aurora's LPC is one real-world case

> **This lesson's main deliverable**: `classify_system` + the three fates. LPC / least squares are the motivation and the closing note; you're not expected to derive the filter in this lesson.

**Why it matters for Aurora**: later, LPC models sound as `Ax=b` (x holds the coefficients); when there are more equations than unknowns, we fall back on least squares.
Whether a system is "unique / infinite / unsolvable" comes down first to `rank` — this lesson nails down the rank test.

← **Previous lesson** · [L14 · Eigenvalues and SVD](L14_eigen_svd.ipynb)

> Last lesson covered **eigenvalues and SVD**: finding the directions a matrix prefers, and using SVD to open up the low-rank world.
> This lesson explores **Gaussian elimination**.

## This lesson's story: you know the destination — work out which road you took

You know the result of some linear transformation (`b`), and you know the transformation's rule (the matrix `A`).
The question: what was the original input `x`? That's `Ax = b`.

It's everywhere in real life:
- (An idealized example) Speech enhancement: given a recording b and some linear model A, work back to the clean speech x — we'll meet this again in signal processing; for today, treat it as an Ax=b story
- Linear regression: given data b and a feature matrix A, find the weight vector x = argmin‖Ax-b‖
- Circuit analysis: given voltages b and an impedance matrix A, solve for the currents x

A system of equations can meet one of three fates:
- **A unique solution**: A has full rank, `rank(A) = rank([A|b]) = n`
- **Infinitely many solutions**: A's columns are linearly dependent (and b is still reachable), so elimination leaves free variables
- **No solution**: b isn't in A's column space, and the equations contradict each other

In this lesson you'll implement `classify_system(A, b)` to decide automatically which case you're in.

## Quick detour: what exactly are "rank" and "column space"?

Before going on, let's stop and make two words that will keep coming up perfectly clear — **rank** and **column space**. If they aren't clear now, judgments like "rank(A) < rank([A|b])" later on will read like gibberish.

**First, an analogy**: suppose three friends each give you directions to the same place. If the three routes actually describe the same road (just worded differently, like "turn left, then go straight" and "before going straight, turn left first"), you've really only received 1 piece of "truly independent" information, even though it sounds like 3. That's what a matrix's **rank** does — it counts how many of its rows (or columns) are "truly independent" and bring new information; ones that repeat, or can be pieced together from the others, don't count.

In numbers:
- A = [[1,0],[0,1]]: the two columns [1,0] and [0,1] can't be built from each other, so both are independent → rank(A) = 2.
- A = [[1,1],[1,1]]: both columns are [1,1]; column 2 = 1 × column 1, adding no new information at all → rank(A) = 1, even though the matrix is 2×2.

**What the column space is**: take all of A's column vectors, add them up with any weights you like (any real coefficients), and the set of every point you can land on is A's "column space." Picture it like this — you're standing at the origin, each column is a direction you're allowed to walk, you can walk any distance along those directions (forwards or backwards), and you can also combine several directions into one step. Everywhere you can reach is the column space. Think of the column space as a "sheet of paper" (or a "line") you can reach: if the target point b isn't on that sheet, no amount of walking on the sheet will get you there.

- If A has a single column [1, 2], its column space is all the points that one direction can span: `{t·[1,2] : t∈ℝ}`, a line through the origin with slope 2 — you can never step off that line.
- If A = [[1,1],[1,1]] (two identical columns), then whatever values x₀ and x₁ take, `x₀·[1,1] + x₁·[1,1] = (x₀+x₁)·[1,1]` — however you combine them, you always land on the line "y=x". Here rank(A)=1, and the column space is just a line, not the whole plane.

Now we can answer "why there's no solution when b isn't in the column space": `Ax=b` asks "can you build b out of A's columns?" If b doesn't lie on the line (or plane) the column space marks out, no choice of x can build it — that's "no solution." The code below checks this: `b=[2,3]` isn't on the "y=x" line (2≠3), so it can't be built; `b=[2,2]` is on the line (2=2), so it can (and because this line has only 1 independent direction but there are 2 coefficients to adjust, there's more than one way to build it — exactly where "infinitely many solutions" comes from).

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

A_dep = np.array([[1., 1.],
                   [1., 1.]])   # two identical columns → rank=1, the column space is just the line y=x

for label, b_test in [('b=[2,3] (not on the line y=x)', np.array([2., 3.])),
                       ('b=[2,2] (on the line y=x)',   np.array([2., 2.]))]:
    rank_A  = np.linalg.matrix_rank(A_dep)
    rank_Ab = np.linalg.matrix_rank(np.column_stack([A_dep, b_test]))
    print(f'{label}: rank(A)={rank_A}, rank([A|b])={rank_Ab}')

print()
print('First case: rank(A)=1 < rank([A|b])=2 → b brings in a new direction → no solution')
print('Second case: rank(A)=1 = rank([A|b])=1 → b brings no new direction, and there is only 1 independent direction but 2 coefficients → infinitely many ways to build it')


## 1. Ax=b: building b out of a linear combination of the columns

Look at `A @ x = b` with a "flip of perspective":

**Column view**: `A @ x = x[0]·col₀ + x[1]·col₁ + ... + x[n-1]·colₙ₋₁`
→ solving `Ax=b` = finding a set of coefficients `x` such that the weighted combination of A's columns equals `b` exactly

**Example**: A = [[2, 1], [0, 1]], b = [5, 3]
Question: can x[0]×[2,0] + x[1]×[1,1] build [5,3]?
Answer: x[0]=1, x[1]=3 → 1×[2,0] + 3×[1,1] = [2,0]+[3,3] = [5,3] ✓

**What Gaussian elimination does**: using elementary row operations, it turns the augmented matrix [A|b] into row echelon form (REF), then solves for the unknowns one by one starting from the last row ("back substitution").

**Rank tells you solvability**:
- `rank(A) = rank([A|b]) = n` → a unique solution
- `rank(A) = rank([A|b]) < n` → infinitely many solutions
- `rank(A) < rank([A|b])` → no solution

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
x = np.array([3.0, 2.0])
b = A @ x

print('column 1 of A:', A[:, 0])
print('column 2 of A:', A[:, 1])
print('x =', x)
print('b = 3 * col1 + 2 * col2 =', b)


## 1.1 Moving up to a three-unknown example

The three-unknown system below is just a bigger version of the same thing: `A` has three columns, `x` has three coefficients, and the goal is to build the `b` on the right.


$$2x + y - 2z = 10,\quad 3x + 2y + 2z = 1,\quad 5x + 4y + 3z = 4$$

The course notes' answer: the unique solution $x=1,\,y=2,\,z=-3$.


## Notation first: look at the shapes, then the operations

`A` is an `(m, n)` matrix, `x` is an `(n,)` vector, and `A @ x` gives an `(m,)` vector. `m` is the number of equations and `n` the number of unknowns; how these two numbers compare decides whether the system is overdetermined, square, or underdetermined.

In [ ]:
import numpy as np
A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10,1,4], float)
print('solution =', np.linalg.solve(A, b), ' (course notes: [1, 2, -3])')

## Hands-on: shapes first, then values

Run this code and pay attention to how `A.shape`, `x.shape` and `b.shape` relate — the actual values of `m` and `n` decide how many equations and unknowns the linear system has, and also whether `np.linalg.solve` can be used to solve it.

In [ ]:
import numpy as np

# Use numpy to watch how the rank changes for matrices of different rank (classifying the solutions is left for exercise 4)
cases = [
    ('unique solution', np.array([[2.,1.],[0.,3.]]), np.array([5.,6.])),
    ('infinitely many', np.array([[1.,2.],[2.,4.]]), np.array([3.,6.])),
    ('no solution', np.array([[1.,1.],[1.,1.]]), np.array([2.,3.])),
]
for label, A, b in cases:
    rank_A  = np.linalg.matrix_rank(A)
    rank_Ab = np.linalg.matrix_rank(np.column_stack([A, b]))
    n = A.shape[1]
    print(f'{label}: rank(A)={rank_A}, rank([A|b])={rank_Ab}, n={n}')
# Observe: the three lines of numbers match the three cases; exercise 4 encodes these comparisons into classify_system.


## Quick detour: why does adding a column change the rank? — Going through the three examples above one by one

The three lines of numbers printed above hold the whole secret of "no solution vs. a solution." The rule is actually very plain: add the column b to A, and the rank either stays the same or goes up by exactly 1 — no more, no less.

Why? Think of "rank" as "how many dimensions the column space currently fills." A's column space already spans some region (a line, a plane…). Now add b as one more column:
- If b could already be built from a linear combination of A's columns (b already "lives" in A's column space), then b brings no new direction, and the rank stays the same.
- If b can't be built, it points in a brand-new direction that A's columns never covered, and the rank goes up by exactly 1 (one more new direction — but only one).

Back to the three examples above:
- **Unique solution**: A=[[2,1],[0,3]]; the two columns [2,0] and [1,3] point different ways and independently fill the whole plane (rank(A)=2). Since the plane is already full, b=[5,6] can naturally be built too, and rank([A|b]) is still 2 — the rank didn't grow, because b couldn't bring a "new direction." And because the two directions were independent to begin with, there's only one way to build it.
- **Infinitely many solutions**: A=[[1,2],[2,4]]; the two columns [1,2] and [2,4] are really the same direction (column 2 = 2 × column 1), rank(A)=1, and the column space is just a line. b=[3,6] happens to lie on that line too ([3,6]=3×[1,2]), so rank([A|b]) is still 1 — no new direction brought in. But this line has only 1 independent direction, and you're using 2 coefficients to hit it, so there are infinitely many mixes (for example x=(3,0) or x=(1,1) both work; check: 1×[1,2]+1×[2,4]=[3,6] ✓).
- **No solution**: A=[[1,1],[1,1]], rank(A)=1, and the column space is the line "y=x". b=[2,3] isn't on that line, so it points in a brand-new direction, and rank([A|b]) becomes 2 — the rank grew, which says b can't be built from A's columns no matter what.

One line to remember: **if rank([A|b]) is bigger than rank(A), b has brought in "new information" that A's columns can't provide — so there's no solution.**

## Code experiment: solve for x, then check the residual

Solve for `x` with `np.linalg.solve`, then compute the residual `‖Ax−b‖` — close to 0 means the solution is right and `A @ x` really lands back on `b`.

In [ ]:
import numpy as np

# After elimination, check the residual of Ax=b
A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10.,1.,4.])
x = np.linalg.solve(A, b)
residual = np.linalg.norm(A @ x - b)
print(f'solution x = {np.round(x,6)}')
print(f'residual ||Ax-b|| = {residual:.2e}  (close to 0 = correct)')
print(f'check: A@x = {np.round(A@x,6)}')


## 1.2 `np.linalg.solve` gives the answer; Gaussian elimination explains the process

First confirm the answer with the tool, then watch the augmented matrix change shape step by step. That way the row operations that follow have a clear goal: separate out the unknowns one at a time.


## 2. Elementary row operations ER1 / ER2 / ER3 and the augmented matrix

- **ER1** swap two rows  `Ri ↔ Rj`
- **ER2** multiply a row by a nonzero constant  `Ri → k·Ri`
- **ER3** add k times another row to a row  `Ri → Ri + k·Rj`

Below we do by hand the step from the course notes: `R2→2R2−3R1`, `R3→2R3−5R1`, clearing everything below the first entry of column 1 to 0.


## Quick detour: why don't these three row operations change the system's solutions?

These three operations can look like "messing about at random" with the matrix, but each one keeps the same promise: **the x values that make the system true are exactly the same before and after — not one more, not one fewer**. Let's go through them one at a time.

**ER1 (swap two rows)**: swapping two rows only changes the order in which you "read" the equations; the equations themselves don't change by a single symbol. Clearly it can't affect which x satisfies them all.

**ER2 (multiply a row by a nonzero constant k)**: this is the same as multiplying both sides of an equation by the same number. For example, multiply both sides of `2x+y=5` by 3 to get `6x+3y=15` — any (x,y) that makes the original equation true also makes the new one true (both sides got 3 times bigger, and the equality still holds); conversely, dividing both sides of the new equation by 3 gives back the original, so this step can be undone and the solution set doesn't change at all (as long as k≠0; multiplying both sides by 0 would turn the equation into `0=0` and throw away the original constraint for nothing — which is exactly why ER2 insists on "nonzero").

**ER3 (add k times another row to a row) — the one that raises the most doubts**: `R2 → R2 + k·R1`, translated back into equations: if the original equation 1 is `a₁·x = c₁` and equation 2 is `a₂·x = c₂`, the new "equation 2" is a weighted sum of the two: `(a₂+k·a₁)·x = c₂+k·c₁`.

The key question: why does this new equation neither add an extra constraint nor lose an original one?

- Suppose x satisfies both original equation 1 and equation 2 (that is, `a₁·x=c₁` and `a₂·x=c₂`). Multiply these two equalities by k and by 1 and add them: `a₂·x + k·a₁·x = c₂ + k·c₁`, which is exactly the new equation — so the original solutions satisfy the new equation too.
- Conversely, if x satisfies equation 1 (unchanged) and the new equation 2', can we conclude it also satisfies the old equation 2? Yes: subtract k times equation 1 from the new equation 2' and you get the old equation 2 back exactly. This step can be undone ("adding it in" and "subtracting it back out" are inverse operations), so no information is lost.

So ER3 isn't "changing" the system; it's using the old equations to "recombine" an equivalent statement — like knowing "Esther's age" and "Esther is 3 years older than Daniel," and concluding "Daniel is 3 years younger than Esther": a restatement that doesn't change the original answer about their ages.

Below we check this concretely with a two-unknown example, `x+y=3` and `2x-y=0`: first solve for the true solution, then apply one ER3 operation and check that the new system gives exactly the same solution.

In [ ]:
import numpy as np

# Original system: x + y = 3,  2x - y = 0
eq1 = np.array([1., 1., 3.])   # [coefficient of x, coefficient of y, right-hand side]
eq2 = np.array([2., -1., 0.])

A_old = np.array([eq1[:2], eq2[:2]])
b_old = np.array([eq1[2], eq2[2]])
x_old = np.linalg.solve(A_old, b_old)
print('solution of the original system (x, y) =', x_old)

# ER3: eq2' = eq2 - 2*eq1  (choose k=-2 to cancel the coefficient 2 of x in eq2)
eq2_new = eq2 - 2 * eq1
print('new equation 2:', eq2_new, ' → i.e. the equation', f'{eq2_new[0]:.0f}x + ({eq2_new[1]:.0f})y = {eq2_new[2]:.0f}')

A_new = np.array([eq1[:2], eq2_new[:2]])
b_new = np.array([eq1[2], eq2_new[2]])
x_new = np.linalg.solve(A_new, b_new)
print('solution of the transformed system (x, y) =', x_new)

print('\nthe two solutions (x, y) match exactly:', np.allclose(x_old, x_new))
print('The new equation is just a "recombination" of the old ones: no new constraint added, no old constraint lost.')


In [ ]:
A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10,1,4], float)
aug = np.column_stack([A, b])      # augmented matrix [A | b]
print('augmented matrix:\n', aug)
aug[1] = 2*aug[1] - 3*aug[0]       # ER3 + ER2
aug[2] = 2*aug[2] - 5*aug[0]
print('\nafter elimination (compare with the extra worked example):\n', aug)


## Quick detour: how were the multipliers 2, 3, 5 chosen during elimination? Why not use fractions?

The code says `aug[1] = 2*aug[1] - 3*aug[0]`, rather than the seemingly more direct `aug[1] = aug[1] - 1.5*aug[0]`. Both versions have the same goal: turn the entry in row 2, column 1 into 0.

Column 1 is currently `[2, 3, 5]` (from rows 1, 2 and 3). To turn the 3 in row 2 into 0:
- **Method A (the one the notebook uses)**: `R2 → 2·R2 − 3·R1`. Check column 1: `2×3 − 3×2 = 0` ✓. This step is really a combination of ER2 (multiply all of R2 by 2) and ER3 (then subtract 3 times R1) — both allowed elementary row operations, so together they're still an undoable operation that doesn't change the solution set.
- **Method B (a more direct single ER3)**: `R2 → R2 − 1.5·R1`. Check: `3 − 1.5×2 = 0` ✓. It also produces a 0, and uses only one operation.

Both methods are legitimate, and they end up at exactly the same solution — after all, whichever coefficients you use, you never step outside the guarantee that "elementary row operations don't change the solution set." Which one to choose is purely an engineering trade-off: method A keeps everything in whole numbers, so it's harder to slip up by hand or in your head, and there's no build-up of rounding error; method B takes fewer steps but brings in decimals (in larger matrices, decimals can pile up precision loss along the way). `np.linalg.solve` uses a more robust algorithm internally (LU decomposition + pivoting), which goes a step further than the whole-number elimination written here in caring about numerical stability, but the basic idea is the same: "each step moves toward making some position 0."

The code below redoes the same system with method B and confirms the solution still matches.

In [ ]:
import numpy as np

A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10,1,4], float)
aug_B = np.column_stack([A, b])

# Method B: a single ER3, fractions allowed
aug_B[1] = aug_B[1] - 1.5*aug_B[0]
aug_B[2] = aug_B[2] - 2.5*aug_B[0]
print('after elimination with method B:\n', aug_B)
print('\ncompare with method A (whole-number version):\n [[2, 1, -2, 10], [0, 1, 10, -28], [0, 3, 16, -42]]')
print('Column 1 becomes [2, 0, 0] either way, and rows 2 and 3 are in the same proportions (method B is method A divided by 2) — two roads, one destination.')


## 2.1 Finishing the elimination: get to "row echelon form," then back-substitute

The last step only handled column 1 (clearing everything below its first entry to 0). The goal of **row echelon form** (REF) is that each row's "leading" nonzero number (called the pivot) sits strictly to the right of the pivot in the row above, so the matrix ends up shaped like a "staircase" — with a whole block of 0s in the lower-left corner. The matrix right now is:

```
[2,  1, -2, 10]
[0,  1, 10, -28]
[0,  3, 16, -42]
```

It isn't a staircase yet — row 3 still has a 3 in column 2, not a 0, so we need to keep eliminating. Use row 2 (its pivot is 1, which is easy to work with) to eliminate in row 3: `R3 → R3 − 3·R2`. After that, row 3 contains only the last unknown, z.

**Back substitution** means: the bottom row of the staircase matrix now contains only one unknown, so solve for it first; once it's solved, substitute it into the row above, which now has only one unknown left and can be solved too; keep going upward like this until you've found the first unknown. It's like peeling an onion one layer at a time — starting from the row with the fewest unknowns.

The code below does both steps and prints every step of the back substitution, alongside the answer from `np.linalg.solve`.

In [ ]:
import numpy as np

A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10,1,4], float)
aug = np.column_stack([A, b])
aug[1] = 2*aug[1] - 3*aug[0]
aug[2] = 2*aug[2] - 5*aug[0]
print('column 1 cleared:\n', aug)

# Keep eliminating: clear row 3, column 2 to 0 as well, giving true row echelon form
aug[2] = aug[2] - 3*aug[1]
print('\ncolumn 2 cleared → row echelon form (REF):\n', aug)

# Back substitution: start from the last row and solve for z, y, x in turn
row3 = aug[2]                       # [0, 0, c, d]  → c*z = d
z = row3[3] / row3[2]
print(f'\nrow 3 {row3[2]:.0f}·z = {row3[3]:.0f}  →  z = {z:.4f}')

row2 = aug[1]                       # [0, b, c, d]  → b*y + c*z = d
y = (row2[3] - row2[2]*z) / row2[1]
print(f'row 2 {row2[1]:.0f}·y + {row2[2]:.0f}·z = {row2[3]:.0f}  →  y = {y:.4f}')

row1 = aug[0]                       # [a, b, c, d]  → a*x + b*y + c*z = d
x = (row1[3] - row1[1]*y - row1[2]*z) / row1[0]
print(f'row 1 {row1[0]:.0f}·x + {row1[1]:.0f}·y + {row1[2]:.0f}·z = {row1[3]:.0f}  →  x = {x:.4f}')

print(f'\nback substitution by hand: x={x:.4f}, y={y:.4f}, z={z:.4f}')
print('np.linalg.solve result:', np.linalg.solve(A, b))
print('They match ✅ — elimination + back substitution is what solve does inside (it just uses a more robust algorithm).')


## 3. The three cases: decide with rank

Compare the rank of the coefficient matrix A, the rank of the augmented matrix [A|b], and the number of unknowns n:
- `rank(A) = rank([A|b]) = n` → **a unique solution**
- `rank(A) = rank([A|b]) < n` → **infinitely many solutions**
- `rank(A) < rank([A|b])` → **no solution** (a contradiction)


## 4. ✏️ Implement `classify_system(A, b)`

**Line of reasoning**:
1. Join A and b into the augmented matrix [A|b], and compute `rA = rank(A)` and `rAb = rank([A|b])`.
2. If `rA < rAb`: b isn't in A's column space, the equations contradict each other, return `'none'`.
3. If `rA == rAb == n`: every unknown has its own independent constraint, return `'unique'`.
4. If `rA == rAb < n`: there aren't enough constraints, free variables exist, return `'many'`.

**Reference input and output**:
- `A=[[2,1],[1,3]], b=[5,10]` → `'unique'` (rank=2=n)
- `A=[[1,2],[2,4]], b=[3,6]` → `'many'` (rank=1<n=2, b is in the column space)
- `A=[[1,2],[2,4]], b=[3,7]` → `'none'` (rank(A)=1 < rank([A|b])=2)

> 💡 Need a hint? Once you've finished the exercise, you can compare with the reference implementation in the `solutions/` folder.


Three things to pin down before writing `classify_system`:
- Input: `A` (the coefficient matrix, `(m, n)`), `b` (the right-hand-side vector, `(m,)`)
- Key steps: compute `rA = rank(A)` and `rAb = rank([A|b])`, then compare them with `n = A.shape[1]`
- Returns: the string `'unique'`, `'many'` or `'none'`

In [ ]:
def classify_system(A, b):
    A = np.asarray(A, float); b = np.asarray(b, float)
    # ✏️ TODO: use rank to decide which case the system is in, and return 'unique'/'many'/'none'
    raise NotImplementedError(
        "TODO: implement classify_system → return 'unique'/'many'/'none'"
    )


In [ ]:
# Three extra worked examples (3×3) + an overdetermined example (4×2, matching Aurora's overdetermined LPC setting)
U  = (np.array([[2,1,-2],[3,2,2],[5,4,3]],float), np.array([10,1,4],float))
M  = (np.array([[1,2,-3],[2,-1,4],[4,3,-2]],float), np.array([6,2,14],float))
N  = (np.array([[1,2,-3],[3,-1,2],[5,3,-4]],float), np.array([-1,7,2],float))
# Overdetermined (4 equations, 2 unknowns): rank=2=n, b is in the column space → 'unique'
OD = (np.array([[1.,0.],[0.,1.],[1.,1.],[2.,1.]]), np.array([1.,1.,2.,3.]))

try:
    print('example 1 (unique):', classify_system(*U),  '  expected: unique')
    print('example 2 (many):  ', classify_system(*M),  '  expected: many')
    print('example 3 (none):  ', classify_system(*N),  '  expected: none')
    print('overdetermined (unique):', classify_system(*OD), '  expected: unique (exactly consistent)')
    assert classify_system(*U)  == 'unique', 'example 1 failed'
    assert classify_system(*M)  == 'many',   'example 2 failed'
    assert classify_system(*N)  == 'none',   'example 3 failed'
    assert classify_system(*OD) == 'unique', 'overdetermined example failed'
    print('\n✅ Passed: solution-type checks done for the three extra examples + the overdetermined example.')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  Task not finished: {e}')
    print('Implement the classify_system function above first, then run this cell.')


### Reference check: the core logic of `classify_system` (open after you've done the exercise)

> Below is a runnable reference implementation for checking your work after the exercise. Please don't peek before you've finished.


In [ ]:
# ── Reference implementation (for comparison after the exercise only) ──
def _classify_system_ref(A, b):
    A  = np.asarray(A, float); b = np.asarray(b, float)
    rA  = np.linalg.matrix_rank(A)
    rAb = np.linalg.matrix_rank(np.column_stack([A, b]))
    n   = A.shape[1]
    if rA < rAb:
        return 'none'
    elif rA == rAb == n:
        return 'unique'
    else:
        return 'many'

# Check that the reference implementation itself is correct
assert _classify_system_ref(*U)  == 'unique'
assert _classify_system_ref(*M)  == 'many'
assert _classify_system_ref(*N)  == 'none'
assert _classify_system_ref(*OD) == 'unique'
print('Reference implementation self-check passed ✅')


**🔗 Aurora connection**: the normal equations of linear regression, `(XᵀX)w = Xᵀy`, are a linear system; solvability corresponds to when the model has a unique best solution.

**What the extra worked examples cover**: the augmented matrix, echelon form, elementary row operations, and classifying solutions.


## Quick detour: how can 4 equations in 2 unknowns still have a "unique solution"?

Intuitively, the more equations and the fewer unknowns, the harder it should be to satisfy them all — either they fight each other (no solution), or at least the answer can't get any "more unique." Yet the overdetermined example `OD` above printed `unique`, and the secret lies in the line `rank(A)=2=n`.

First, look at what these four equations actually say. `OD`'s `A` and `b` are:

```
[1, 0]        [1]
[0, 1]   x =  [1]
[1, 1]        [2]
[2, 1]        [3]
```

In plain words: `x=1` (row 1), `y=1` (row 2), `x+y=2` (row 3), `2x+y=3` (row 4). The first two rows already pin down both x and y — no more information needed. Rows 3 and 4 look like "extra" equations, but plug in x=1, y=1 and check: `x+y=1+1=2` ✓, `2x+y=2+1=3` ✓ — they agree exactly with the answer from the first two rows, adding no new constraint and creating no contradiction.

In the language of rank: row 3 is really "row 1 + row 2," and row 4 is "2×row 1 + row 2" — they're **linear combinations** of the first two rows and point in no new direction, so `rank(A)` is still only 2 (not 4). At the same time, because the 3rd and 4th values in b satisfy the same linear-combination relationships (`2=1+1`, `3=2×1+1`), the rank of the augmented matrix is also still 2 — there's no contradiction where "a row adds a new direction but b doesn't follow." `rank(A)=rank([A|b])=n=2`, which lands squarely in the "unique solution" case.

**This is a deliberately "tame" overdetermined system.** In the more common overdetermined setting (like Aurora's LPC: equations from hundreds of samples, but only a dozen or so filter coefficients), the extra equations usually *don't* line up exactly — the measured equations contradict each other slightly because of noise, so strictly speaking there is "no solution" (`rank(A) < rank([A|b])`). Then "unique solution" has to be redefined as "the unique solution in the least-squares sense": not "an x that satisfies every equation exactly" (there isn't one), but "the x that makes the sum of squared errors of all the equations, `‖Ax-b‖²`, as small as possible" — and that x is usually uniquely determined, thanks to the `lstsq` mentioned earlier.

The code below checks that "rows 3 and 4 are linear combinations of the first two rows," and also builds an "untamed" overdetermined example to see what `lstsq` does with a genuinely contradictory overdetermined system.

In [ ]:
import numpy as np

A_od, b_od = OD
print('row 3 = row 1 + row 2?', np.allclose(A_od[2], A_od[0] + A_od[1]),
      ' b[2]=1+1?', b_od[2] == b_od[0] + b_od[1])
print('row 4 = 2*row 1 + row 2?', np.allclose(A_od[3], 2*A_od[0] + A_od[1]),
      ' b[3]=2+1?', b_od[3] == 2*b_od[0] + b_od[1])
print(f'rank(A) = {np.linalg.matrix_rank(A_od)}  (not 4, because the last two rows add no new direction)')

print()
print('--- For comparison: an "untamed" overdetermined system (4 equations, 2 unknowns, but noisy and contradictory) ---')
rng = np.random.default_rng(0)
A_noisy = np.array([[1.,0.],[0.,1.],[1.,1.],[2.,1.]])
b_noisy = np.array([1., 1., 2., 3.]) + rng.normal(0, 0.05, size=4)  # add a tiny bit of noise
rank_noisy    = np.linalg.matrix_rank(A_noisy)
rank_noisy_ab = np.linalg.matrix_rank(np.column_stack([A_noisy, b_noisy]))
print(f'rank(A)={rank_noisy}, rank([A|b])={rank_noisy_ab}  → strictly speaking this is already "none" (no exact solution)')
x_best, res, rank_, sv_ = np.linalg.lstsq(A_noisy, b_noisy, rcond=None)
print(f'lstsq\'s "unique solution in the least-squares sense": x={np.round(x_best,4)}, residual={np.linalg.norm(A_noisy@x_best-b_noisy):.4f}')
print('This is exactly the real situation of Aurora\'s LPC coefficient estimation: more equations than unknowns, and noise keeps them from agreeing exactly,')
print('yet lstsq still gives a definite, reproducible, minimum-error answer.')


## 🎨 Picture: Ax=b as a linear combination of A's columns (extra worked example 1)


In [ ]:
from laviz import style, mat_times_vec
style()
mat_times_vec([[2,1,-2],[3,2,2],[5,4,3]],[1,2,-3]);  # = b=[10,1,4]

## Quick detour: what exactly does the condition number measure?

Start with an everyday scene: cut paper with a very blunt pair of scissors, and the slightest wobble of your hand sends the cut way off line; switch to a sharp, steady pair, and the same wobble barely affects the cut. The **condition number** measures how sensitive the matrix "machine" is to "input wobble" — numerically, the larger `κ(A)` is, the more a tiny error in `b` (even one that comes only from floating-point rounding) gets amplified by the solving process into a much larger error in `x`.

Why does that happen? Go back to the "column space" picture: if two rows (or columns) of A point in almost the same direction (like `[1,1]` and `[1,1+eps]` in the example below, which nearly coincide when eps is small), then the two equations are almost "the same sentence said twice," and there's very little "independent information" to really pin x down — a tiny nudge can send the "intersection point" a long way off (picture two nearly parallel lines: turn one by a hair and the intersection jumps far away; if the two lines cross at a wide angle, a small turn barely moves the intersection). A larger `κ(A)` means "these two rows are closer to parallel," and the system is more sensitive to noise.

The formal definition of the condition number is `κ(A) = ‖A‖·‖A⁻¹‖` (the matrix norm times the norm of its inverse), and you can also compute it with the singular values from L14: `κ(A) = σ_max / σ_min` (the largest singular value divided by the smallest). `κ(A)=1` is the ideal case (for example the identity matrix, which doesn't amplify errors at all); the further `κ(A)` is above 1, the more errors get amplified; and `κ(A) → ∞` means the matrix has become singular (coming up right below).

Talk alone isn't very convincing — below we first repeat the original experiment (watching how κ(A) changes with eps), then add one key demonstration: **add the same tiny bit of noise to b, and for a matrix with small κ(A) x barely changes, while for a matrix with large κ(A) x jumps a long way**.

In [ ]:
import numpy as np

# Parameter experiment: the closer A's row vectors are to linear dependence, the more the condition number explodes
for eps in [0.1, 0.01, 0.001, 1e-6]:
    A = np.array([[1., 1.],[1., 1.+eps]])
    kappa = np.linalg.cond(A)
    print(f'eps={eps:.0e}  κ(A)={kappa:.2e}')
print('→ as eps→0 the row vectors become nearly parallel, and solving Ax=b numerically gets less and less stable.')


In [ ]:
import numpy as np

# Comparison: a matrix with a small condition number vs. one with a large condition number — for the same small nudge to b, how different is the change in x?
A_good = np.array([[1., 0.], [0., 1.]])          # identity matrix, κ=1, the most stable
A_bad  = np.array([[1., 1.], [1., 1.0001]])       # two nearly parallel rows, κ is large

b = np.array([2., 2.])
noise = np.array([0.001, -0.001])                # the same small nudge

for label, Amat in [('κ(A)≈1 (well-conditioned)', A_good), (f'κ(A)≈{np.linalg.cond(A_bad):.0e} (ill-conditioned)', A_bad)]:
    x0 = np.linalg.solve(Amat, b)
    x1 = np.linalg.solve(Amat, b + noise)
    shift = np.linalg.norm(x1 - x0)
    print(f'{label}: with b nudged by 0.001, x changes by = {shift:.4f}')

print('\nConclusion: the nudge to b is exactly the same size, but the ill-conditioned matrix (large κ) amplifies the error by several orders of magnitude —')
print('that is the real meaning of "large condition number → numerically unstable": it is not that no solution comes out, but that the solution may be completely untrustworthy.')


## Quick detour: what is a determinant? And what does "singular matrix" mean?

The **determinant** is an operation that "compresses" a square matrix into a single number, and that number tells you how much the linear transformation the matrix represents scales areas (or volumes) in space, and whether it flips orientation.

- For a 2×2 matrix `A=[[a,b],[c,d]]`, the determinant is defined as `det(A) = a·d − b·c`. Geometrically, it's the area of the parallelogram spanned by A's two column vectors `[a,c]` and `[b,d]` (with a sign — if the two directions have been "flipped," the area counts as negative).
- For a 3×3 matrix, the determinant is the volume of the parallelepiped spanned by its three column vectors; the formula is longer, but the meaning is the same.

**Why `det(A)=0` means "singular"**: if the determinant is 0, the "parallelogram/parallelepiped" has been squashed flat — its area or volume has become 0. In other words, A's column vectors no longer each span out a "new" region of their own; they're crowded onto a line (or even a point) — exactly the geometric picture of "linearly dependent columns," `rank(A) < n`. The matrix squashes the original space into something of lower dimension, and that squashing can't be "walked back" (you can't restore a flattened sheet of paper into a cube), so A has no inverse, and `np.linalg.solve` can't find a unique solution — it can only raise `LinAlgError: Singular matrix`.

Example: `A=[[1,2],[2,4]]`, `det(A) = 1×4 − 2×2 = 0` — singular, as expected. It's also the very matrix this lesson's "infinitely many / no solution" examples keep using: its two columns `[1,2]` and `[2,4]` point the same way (the second is twice the first), so they can't span a plane, only a line — confirming what we said earlier about "the column space collapsing to a line."

The code below checks this 2×2 determinant formula.

In [ ]:
import numpy as np

A_demo = np.array([[1., 2.], [2., 4.]])
a, b_, c, d = A_demo[0,0], A_demo[0,1], A_demo[1,0], A_demo[1,1]
det_manual = a*d - b_*c
det_numpy  = np.linalg.det(A_demo)
print(f'det(A) by hand = a*d - b*c = {a}*{d} - {b_}*{c} = {det_manual}')
print(f'np.linalg.det(A) = {det_numpy:.10f}  (approximately 0, up to floating-point error)')
print('→ det(A)=0: the two columns are collinear and cannot span a plane → singular matrix, no inverse.')


## Parameter experiment: how a singular matrix triggers an error, and how lstsq comes to the rescue

Set A's determinant to 0 (e.g. `A=[[1,2],[2,4]]`), and calling `np.linalg.solve(A, b)` raises `LinAlgError: Singular matrix` — a singular matrix has no inverse, so there's no unique solution to be had. Then hand the same matrix to `np.linalg.lstsq(A, b, rcond=None)`: it doesn't raise an error, and returns the minimum-norm solution (when there are infinitely many) or the approximate solution with the smallest residual (when the equations contradict each other). Change only one thing at a time, predict first whether `solve` or `lstsq` will succeed, then run it to check.

In [ ]:
import numpy as np

A_sing = np.array([[1., 2.],
                   [2., 4.]])   # determinant = 0, a singular matrix

# ── Scenario 1: consistent equations (b is in the column space) ──
b_consistent   = np.array([3., 6.])  # [3,6] = 3*[1,2]
b_inconsistent = np.array([3., 7.])  # [3,7] is not in the space spanned by [1,2]

print('=== np.linalg.solve (should raise LinAlgError for a singular matrix) ===')
for label, bv in [('consistent', b_consistent), ('contradictory', b_inconsistent)]:
    try:
        x = np.linalg.solve(A_sing, bv)
        print(f'  {label} b: unexpectedly succeeded, x={x}')
    except np.linalg.LinAlgError as e:
        print(f'  {label} b: ✅ caught LinAlgError — {e}')

print()
print('=== np.linalg.lstsq (no error; returns the minimum-norm / minimum-residual solution) ===')
for label, bv in [('consistent', b_consistent), ('contradictory', b_inconsistent)]:
    x_ls, res, rank, sv = np.linalg.lstsq(A_sing, bv, rcond=None)
    residual = np.linalg.norm(A_sing @ x_ls - bv)
    print(f'  {label} b: x_ls={np.round(x_ls,4)},  residual={residual:.4f},  rank={rank}')
print()
print('Conclusion: solve fails on a singular matrix; lstsq always gives an answer,')
print('      but the residual is ≈0 for consistent equations and >0 for contradictory ones — classify_system can tell you in advance.')


## Quick detour: the 4 values `lstsq` returns, and which answer the "minimum-norm solution" / "minimum-residual solution" actually picks

The code writes `x_ls, res, rank, sv = np.linalg.lstsq(A, b, rcond=None)`. What each of the four names means:

1. **`x_ls`**: the solution vector it finally gives — the only one of the 4 return values you'll usually use.
2. **`res`**: the sum of squared residuals `‖Ax−b‖²` (it only comes back as a non-empty array when the system is "overdetermined with no exact solution"; when the system has an exact solution or is underdetermined, `res` is often an empty array).
3. **`rank`**: the `rank(A)` that `lstsq` computes internally; you can compare it with your own `np.linalg.matrix_rank(A)`, and normally the two should agree.
4. **`sv`**: A's **singular values** (the set of numbers from the SVD covered in L14). There are always min(m, n) singular values (for a square or tall A, that's the number of columns), but **the number of nonzero singular values equals rank(A)** — if a singular value is almost 0 (say `1.8e-16`, zero only up to floating-point error), the matching direction "exists in name only" and provides no truly independent information, which is also why "counting the nonzero singular values" is a practical way to compute the rank.

Now for how the two kinds of "solution" are chosen. Behind each is a "fair default choice":

- **Infinitely many solutions → the minimum-norm solution**: when the system has infinitely many solutions (like earlier, where both `x=(3,0)` and `x=(1,1)` satisfy `[1,2]x₀+[2,4]x₁=[3,6]`), `lstsq` doesn't just pick one at random; among all the solutions it picks the one with the **smallest norm (the vector length from L11)** — think of it as "without breaking any constraint, choose the least effortful, least extreme answer," like having countless ways to push a shopping cart to the checkout, with `lstsq` choosing the shortest path by default.
- **No solution → the minimum-residual solution**: when the system has no solution (b isn't in the column space, and no choice of x can make `Ax` exactly equal b), `lstsq` instead looks for an x that makes `‖Ax−b‖` (the distance between "the guessed answer" and "the real target") as small as possible — that's "minimum residual." This x is uniquely determined (as long as A has full column rank), because "the point in the column space closest to b" is a single, unique projection point (picture dropping a perpendicular from the point b onto the "wall" of the column space: there's only one foot of the perpendicular).

The code below prints both `sv` (the singular values) and `res` (the residual), so you can compare the "consistent" and "contradictory" cases.

In [ ]:
import numpy as np

A_sing = np.array([[1., 2.], [2., 4.]])
b_consistent   = np.array([3., 6.])
b_inconsistent = np.array([3., 7.])

for label, bv in [('consistent (b=[3,6])', b_consistent), ('contradictory (b=[3,7])', b_inconsistent)]:
    x_ls, res, rank, sv = np.linalg.lstsq(A_sing, bv, rcond=None)
    print(f'--- {label} ---')
    print(f'  x_ls = {np.round(x_ls, 4)}   (minimum-norm solution / minimum-residual solution)')
    print(f'  res  = {res}   (non-empty only when "contradictory and overdetermined")')
    print(f'  rank = {rank}   (compare np.linalg.matrix_rank = {np.linalg.matrix_rank(A_sing)})')
    print(f'  sv   = {np.round(sv, 6)}   (singular values; number of nonzero ones = rank)')
    print(f'  residual by hand ‖Ax-b‖ = {np.linalg.norm(A_sing @ x_ls - bv):.6f}')
    print()

print('In sv, one value is about 5.0 and the other is almost 0 — only 1 "truly effective" direction, matching rank(A)=1.')


## Quick detour: should you use `solve` or `lstsq`?

A simple decision flow that ties together everything in this lesson:

1. First run `classify_system(A, b)` (or compare `rank(A)` with `rank([A|b])` and `n` directly).
2. If the result is `'unique'`: `np.linalg.solve(A, b)` works directly and gives the unique exact solution.
3. If the result is `'many'`: `solve` will usually raise an error or give unstable results (it assumes a unique solution exists); switch to `np.linalg.lstsq(A, b)` — out of the infinitely many solutions it returns the "minimum-norm" one, a reproducible default answer chosen by a clear rule.
4. If the result is `'none'`: `solve` will definitely raise an error or give a wrong answer, and `lstsq` is the only option — it gives the "closest" approximate solution (the minimum-residual solution). This is also why Aurora's LPC coefficient estimation uses least squares in practice: the overdetermined system built from a real recording (hundreds of samples, only a dozen or so coefficients) is almost never exactly consistent, and least squares still gives a meaningful, usable answer when "there is no exact solution."

## Wrap-up

You can now use `np.linalg.solve(A, b)` to find a unique solution, and `classify_system(A, b)` to decide whether a system can be solved. Aurora's LPC coefficient estimation takes the autocorrelation matrix of a recording segment as `A` and solves this linear system to get the prediction filter `x`; when `rank(A)` falls short, it has to switch to the least-squares form. The next lesson introduces determinants and inverses, giving a geometric explanation of how a system degenerates when its `rank` drops.

---
⬇️ **Checkpoint**: you've read the wrap-up; now do the whiteboard challenge below before ticking off the self-review.


## ✏️ Whiteboard challenge: linear systems by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: the system `2x + y = 5, x + 3y = 8`
Write down the augmented matrix `[A|b]`, eliminate, and find x and y.

**Question 2**: A = [[1, 2], [2, 4]], b = [1, 3]
Does this system have a unique solution, infinitely many, or none? (Compute rank(A) and rank([A|b]) by hand)

**Question 3**: A = [[1, 2], [2, 4]], b = [2, 4]
Exactly the same matrix as Question 2, but b has changed — which case is it now?

**Question 4**: in a linear system `Ax=b`, if A is a 4×4 matrix with `rank(A) = 4`,
how many solutions does the system have, for any b?

When you've finished, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: 2x+y=5, x+3y=8
A1 = np.array([[2., 1.], [1., 3.]])
b1 = np.array([5., 8.])
x1 = np.linalg.solve(A1, b1)
assert np.allclose(A1 @ x1, b1, atol=1e-10)
print(f"Q1 ✅  2x+y=5, x+3y=8 → x={x1[0]:.4f}, y={x1[1]:.4f}")
print(f"       check A@x = {np.round(A1 @ x1, 4)} = b = {b1}")

# Q2: decide by rank (A's rows are linearly dependent, b contradicts them)
A2 = np.array([[1., 2.], [2., 4.]])
b2_no = np.array([1., 3.])
rA2 = np.linalg.matrix_rank(A2)
rAb2 = np.linalg.matrix_rank(np.column_stack([A2, b2_no]))
print(f"\nQ2 ✅  rank(A)={rA2}, rank([A|b])={rAb2} → no solution (the ranks differ, b is not in the column space)")
try:
    cs = classify_system(A2, b2_no)
    assert 'no' in cs.lower() or 'none' in cs or 'inconsistent' in cs.lower()
    print(f"       classify_system → '{cs}'")
except (NotImplementedError, TypeError):
    print(f"       (classify_system not implemented yet)")

# Q3: same A, b becomes [2,4] (in the column space)
b2_inf = np.array([2., 4.])
rAb3 = np.linalg.matrix_rank(np.column_stack([A2, b2_inf]))
print(f"\nQ3 ✅  rank(A)={rA2}, rank([A|b])={rAb3}, n=2 → rank(A)<n → infinitely many solutions")
try:
    cs3 = classify_system(A2, b2_inf)
    assert 'inf' in cs3.lower() or 'infinite' in cs3 or 'many' in cs3.lower()
    print(f"       classify_system → '{cs3}'")
except (NotImplementedError, TypeError):
    print(f"       (classify_system not implemented yet)")

# Q4: 4×4 full rank → unique solution
A4 = np.eye(4) * 2 + np.random.default_rng(0).standard_normal((4,4)) * 0.1
rank_A4 = np.linalg.matrix_rank(A4)
assert rank_A4 == 4, f"the matrix should have rank 4"
print(f"\nQ4 ✅  4×4 full-rank matrix (rank={rank_A4}=n=4) → a unique solution (for any b)")
print("\n🎉 Linear-systems whiteboard challenge passed! Deciding solvability by rank is now yours.")

In [ ]:
# ✏️ Self-review for this lesson
l15_review = {
    "classify_system_implemented": None,  # classify_system implemented and passes its asserts? True/False
    "gaussian_elimination_understood": None,  # understand how elimination on the augmented matrix works? True/False
    "rank_solvability_rule":       None,  # can use rank to tell unique / infinitely many / no solution? True/False
    "column_space_intuition":      None,  # understand what "b is in A's column space" means? True/False
    "whiteboard_passed":           None,  # finished the whiteboard challenge on paper? True/False
}

unfilled = [k for k, v in l15_review.items() if v is None]
assert not unfilled, f'Not filled in yet: {unfilled}'
weak = [k for k, v in l15_review.items() if v is False]
if weak:
    print(f'⚠️  Needs more work: {weak}')
else:
    print('✅ L15 complete! On to L16: determinants and inverses')

---

→ **Next lesson** · [L16 · Determinants and inverses](L16_determinant_inverse.ipynb)

> Next lesson covers **determinants and inverses**: the geometric meaning of det(A) (area scaling), inverting a matrix, and when you can't.